# Personal Expense Tracker with Spending Insights
## Exploratory Data Analysis, Feature Engineering & ML Categorization
**Author:** Khushi Yadav (Final-Year B.Tech CSE - AI & DS)  
**Objective:** End-to-end experimental notebook demonstrating NLP feature extraction (TF-IDF), Multi-Class Classification, statistical anomaly detection (Z-Score & IQR), and Holt-Winters forecasting.

In [1]:
import os
import sys
from pathlib import Path

# Set project root
PROJECT_ROOT = Path(".").resolve().parent if Path(".").resolve().name == "notebooks" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from src.ml.classifier import TransactionClassifier
from src.parser.sms_parser import BankSMSParser
from src.analytics.patterns import SpendingPatternAnalyzer
from src.forecast.forecaster import SpendingForecaster

print("Environment initialized successfully!")

### 1. Load & Inspect Anonymized Transaction Corpus

In [2]:
dataset_path = PROJECT_ROOT / "data" / "processed" / "cleaned_transactions.csv"
df = pd.read_csv(dataset_path)
print(f"Total transactions: {len(df)}")
df.head()

### 2. Category Distribution & Spending Velocity Analysis

In [3]:
cat_summary = df[df["transaction_type"] == "Debit"].groupby("category")["amount"].agg(["count", "sum", "mean"]).reset_index()
cat_summary = cat_summary.sort_values(by="sum", ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=cat_summary, x="sum", y="category", palette="viridis")
plt.title("Total Expenditure by Category (12-Month Period)")
plt.xlabel("Expenditure (INR ₹)")
plt.ylabel("Expense Category")
plt.tight_layout()
plt.show()

### 3. Machine Learning Model Training (TF-IDF + Calibrated Linear Classifier)

In [4]:
X = df["narration"].fillna(df["merchant"]).astype(str).tolist()
y = df["category"].astype(str).tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

clf = TransactionClassifier(confidence_threshold=0.30)
clf.fit(X_train, y_train)

preds = clf.predict_batch(X_test)
y_pred = [p[0] for p in preds]

print("=== Classification Report ===")
print(classification_report(y_test, y_pred, zero_division=0))

### 4. Holt-Winters Time-Series Spending Forecast

In [5]:
forecaster = SpendingForecaster(alpha=0.4, beta=0.2)
forecast_res = forecaster.forecast_total_spending(df)
print(f"Forecast for {forecast_res['forecast_month']}:")
print(f"Predicted Expense: ₹{forecast_res['predicted_expense']:,.2f}")
print(f"80% Confidence Interval: ₹{forecast_res['lower_bound']:,.2f} - ₹{forecast_res['upper_bound']:,.2f}")
print(f"Projected Savings: ₹{forecast_res['predicted_savings']:,.2f}")